In [28]:
# Importing necessary libraries
import pandas as pd
import torch
from torch import nn

In [29]:
# Loading Dataset
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [30]:
# Splitting the data
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

In [31]:
# Scaling
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [32]:
# Label Encoding
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [33]:
# Data loading class
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):
    def __init__(self, features, labels) -> None:
        super().__init__()
        self.features = torch.tensor(features, dtype = torch.float32)
        self.labels = torch.tensor(labels, dtype = torch.long)

    def __len__(self):
        return self.features.shape[0]

    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [34]:
# Creating an object
train_dataset = CustomDataset(X_train, y_train)
test_dataset = CustomDataset(X_test, y_test)

# Data Loaders
train_loader = DataLoader(dataset = train_dataset, batch_size = 32, shuffle = True, pin_memory = True)
test_loader = DataLoader(dataset = test_dataset, batch_size = 32, shuffle = True, pin_memory = True)

In [35]:
import torch.nn.init as init # Used to initialize the parameters

class CustomModel(nn.Module):
    def __init__(self, num_features):
        super(CustomModel, self).__init__()

        # Building model
        self.network = nn.Sequential(
            # For ReLU and its variants - He Initialization
            nn.Linear(in_features = num_features, out_features = 128),
            nn.LeakyReLU(negative_slope = 0.01),

            nn.Linear(in_features = 128, out_features = 64),
            nn.ELU(alpha = 1),

            nn.Linear(in_features = 64, out_features = 32),
            nn.PReLU(num_parameters = 1, init = 0.25),

            # For Tanh and sigmoid - Glorat Initialization
            nn.Linear(in_features = 32, out_features = 16),
            nn.Tanh(),

            nn.Linear(in_features = 16, out_features = 10),
            nn.Sigmoid()
        )

        # Initializing weights
        self.initialize_weights()

    def initialize_weights(self):
        # Find the layer in the sequential network to see what follows
        for i, module in enumerate(self.network):
            if i + 1 < len(self.network) and isinstance(module, nn.Linear):
                next_layer = self.network[i + 1]
                if isinstance(next_layer, (nn.ReLU, nn.LeakyReLU, nn.ELU, nn.PReLU)):
                    init.kaiming_normal_(module.weight, nonlinearity='leaky_relu')
                elif isinstance(next_layer, nn.Sigmoid):
                    init.xavier_normal_(module.weight)
                
                # Constant Initialization - The type of parameters must be torch.nn.Parameter
                """
                    module.weight = torch.nn.Parameter(torch.zeros_like(module.weight, requires_grad=True))
                """

    def forward(self, X):
        return self.network(X)

In [36]:
# Model Object - Weights before optimization
model = CustomModel(num_features = X_train.shape[1])

for i, module in enumerate(model.network):
    if isinstance(module, nn.Linear):
        print(module, module.weight, sep = '\n', end = '\n' + '_' * 50 + '\n')

Linear(in_features=30, out_features=128, bias=True)
Parameter containing:
tensor([[ 0.1372,  0.3515,  0.0704,  ..., -0.7568,  0.0458, -0.0461],
        [ 0.2032,  0.3290,  0.2766,  ...,  0.2981,  0.1387,  0.0934],
        [-0.0458,  0.2109,  0.1207,  ...,  0.2416, -0.3374, -0.1560],
        ...,
        [-0.2241,  0.3321, -0.1715,  ..., -0.5002,  0.0908,  0.0664],
        [ 0.2386,  0.1107,  0.1523,  ...,  0.3156,  0.2299,  0.5110],
        [-0.0100,  0.0805, -0.0475,  ..., -0.0384,  0.1539,  0.4296]],
       requires_grad=True)
__________________________________________________
Linear(in_features=128, out_features=64, bias=True)
Parameter containing:
tensor([[-0.0029,  0.2049,  0.1248,  ...,  0.0209, -0.1606,  0.0088],
        [ 0.1617,  0.1417, -0.2159,  ...,  0.1193,  0.1327,  0.1076],
        [-0.1342,  0.1571, -0.0003,  ...,  0.0375,  0.0541, -0.1129],
        ...,
        [ 0.0429,  0.1750, -0.2217,  ..., -0.0217,  0.0325, -0.0069],
        [-0.1031,  0.1215,  0.0493,  ..., -0.18

In [37]:
# Compiling Model
learning_rate = 0.01
epochs = 10

# Defining Loss Function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = torch.optim.Adam(params = model.parameters(), lr = learning_rate)

In [38]:
# Training Loop
for epoch in range(epochs):
    total_epoch_loss = 0

    for batch_features, batch_labels in train_loader:
        # Forward pass
        y_pred = model(batch_features)

        # Calculate loss
        loss = criterion(y_pred, batch_labels)

        # Zero Gradients
        optimizer.zero_grad()

        # Backward pass
        loss.backward()

        # Update Weights
        optimizer.step()

        # Epoch loss
        total_epoch_loss += loss.item()

    avg_loss = total_epoch_loss/len(train_loader)
    print(f'Epoch: {epoch + 1} , Loss: {avg_loss}')

Epoch: 1 , Loss: 1.836983895301819
Epoch: 2 , Loss: 1.609567681948344
Epoch: 3 , Loss: 1.5285129149754841
Epoch: 4 , Loss: 1.5066572030385335
Epoch: 5 , Loss: 1.496070392926534
Epoch: 6 , Loss: 1.4889601866404216
Epoch: 7 , Loss: 1.4900134722391765
Epoch: 8 , Loss: 1.4842170238494874
Epoch: 9 , Loss: 1.4804006179173788
Epoch: 10 , Loss: 1.4768604596455892


In [39]:
# Values of parameters after optimization
for i, module in enumerate(model.network):
    if isinstance(module, nn.Linear):
        print(module, module.weight, sep = '\n', end = '\n' + '_' * 50 + '\n')

Linear(in_features=30, out_features=128, bias=True)
Parameter containing:
tensor([[ 0.1443,  0.3844,  0.0973,  ..., -0.7071, -0.0787,  0.0645],
        [ 0.0869,  0.2963,  0.1484,  ...,  0.1760,  0.0502, -0.1359],
        [-0.1093,  0.0741,  0.0580,  ...,  0.2133, -0.3664, -0.1595],
        ...,
        [-0.2232,  0.3186, -0.1769,  ..., -0.4573,  0.0169,  0.0932],
        [ 0.0939,  0.0013,  0.0196,  ...,  0.2432,  0.2472,  0.4909],
        [-0.0323,  0.0868, -0.0787,  ...,  0.1158,  0.3123,  0.5259]],
       requires_grad=True)
__________________________________________________
Linear(in_features=128, out_features=64, bias=True)
Parameter containing:
tensor([[ 0.1270,  0.1420,  0.0854,  ...,  0.0892, -0.2169, -0.1786],
        [ 0.1592,  0.1922, -0.1652,  ...,  0.1626,  0.1848,  0.2371],
        [-0.2161,  0.2008,  0.0426,  ..., -0.0283,  0.0937,  0.0127],
        ...,
        [-0.0118,  0.2130, -0.2422,  ..., -0.1182,  0.0503, -0.0706],
        [-0.0546,  0.0256,  0.0950,  ..., -0.12

In [40]:
model.eval()

CustomModel(
  (network): Sequential(
    (0): Linear(in_features=30, out_features=128, bias=True)
    (1): LeakyReLU(negative_slope=0.01)
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ELU(alpha=1)
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): PReLU(num_parameters=1)
    (6): Linear(in_features=32, out_features=16, bias=True)
    (7): Tanh()
    (8): Linear(in_features=16, out_features=10, bias=True)
    (9): Sigmoid()
  )
)

In [41]:
# Evaluating on Test Data - Demo
from torch import tensor
from torchmetrics.classification import MulticlassAccuracy
target = tensor([2, 1, 0, 0])
preds = tensor([2, 1, 0, 1])
metric = MulticlassAccuracy(num_classes=3)
metric(preds, target)

tensor(0.8333)

In [42]:
mca = MulticlassAccuracy(num_classes=3, average=None)
mca(preds, target)

tensor([0.5000, 1.0000, 1.0000])

In [43]:
# Evaluating on Test Data
accuracy = []
with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        y_pred = model(batch_features)
        _, predicted = torch.max(y_pred, 1)
        metric = MulticlassAccuracy(num_classes = 3)
        accuracy.append(metric(predicted, batch_labels))
print(sum(accuracy) / len(accuracy))

tensor(0.9875)
